# 16. 판정요지·지식베이스 불용어 적용 효과 검증

15번에서는 판정요지 문장으로 선정한 불용어를 제품 속성 5개에 적용했지만 예측 결과가 바뀌지 않았습니다. 이는 불용어 후보가 추출된 텍스트와 실제 적용한 텍스트의 성격이 달랐기 때문일 수 있습니다.

이번 실험은 불용어를 추출한 원천과 가까운 텍스트에 적용합니다.

1. **HS 지식베이스 검색:** 지식 문서의 본문에 불용어를 적용하고, 제품 속성 기반 검색 질의는 그대로 둡니다.
2. **과거 판정 사례 검색:** 과거 사례의 판정요지에만 불용어를 적용하고, 제품 속성 기반 검색 질의와 사례의 제품 속성은 그대로 둡니다.

두 실험 모두 불용어 적용 전후의 검색 지표를 비교합니다. 검색 질의에 판정요지나 정답 HS 코드를 넣지 않습니다. 이는 검색 성능 실험이며 HS 코드의 법적 분류 적합성을 자동 확정하는 실험은 아닙니다.

In [ ]:
!pip -q install -U sentence-transformers kiwipiepy

In [ ]:
import re, hashlib
import numpy as np
import pandas as pd
from urllib.parse import quote
from kiwipiepy import Kiwi
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import StratifiedGroupKFold

BASE = 'https://raw.githubusercontent.com/lin1118-code/HS-complete-product/main/'
cases_file = quote('완제품 모델 데이터_사례_전처리 후 - 사례_전처리 완료.csv',safe='')
data_all = pd.read_csv(BASE+cases_file).fillna('')
kb = pd.read_json(BASE+'hs_knowledge_base.jsonl',lines=True).fillna('')
INPUT_COLUMNS = ['Product Name','Main Ingredients (KR)','Main Ingredients (EN)','Product Form and Packaging','Intended Use']
TARGET = 'HS Code'
SUMMARY = 'Preprocessed Ruling Summary'
STOPWORDS = {'것','경우','관세율표','규정','기재','등','따라서','물품','및','본건','본품','분류','설명','소호','수','이상','제품','함','해당','호','표','관세'}
kiwi = Kiwi()

def remove_stopword_morphs(text):
    text = str(text)
    chars = list(text)
    removed = 0
    for token in kiwi.tokenize(text):
        if token.form in STOPWORDS:
            for i in range(token.start,token.start+token.len): chars[i] = ''
            removed += 1
    return ''.join(chars),removed

for col in INPUT_COLUMNS+[TARGET,SUMMARY]:
    data_all[col] = data_all[col].astype(str).replace('nan','').fillna('').str.strip()
data_all = data_all[data_all[TARGET].ne('')].copy()
data_all['query_text'] = data_all[INPUT_COLUMNS].agg(' '.join,axis=1).str.strip()
data_all['duplicate_group'] = data_all[INPUT_COLUMNS].agg('||'.join,axis=1).map(lambda x:hashlib.md5(x.encode()).hexdigest())
eligible = data_all.groupby(TARGET).filter(lambda x:len(x)>=4)
group_counts = eligible.groupby(TARGET)['duplicate_group'].nunique()
labels = sorted(group_counts[group_counts>=4].index.astype(str))
data = eligible[eligible[TARGET].isin(labels)].copy().reset_index(drop=True)
data[TARGET] = data[TARGET].astype(str)

print('평가 사례:',len(data),'| HS 코드:',data[TARGET].nunique(),'| KB 문서:',len(kb))
print('판정요지 결측:',int(data[SUMMARY].eq('').sum()))

## 1. 불용어 적용 범위 및 전처리 진단

불용어는 Kiwi 형태소의 완전 일치 기준으로 제거합니다. 제품 속성 기반 검색 질의는 변경하지 않습니다. 지식베이스에서는 문서 본문만, 과거 사례 검색에서는 판정요지만 전처리합니다.

In [ ]:
kb['document_text'] = kb['document_text'].fillna('').astype(str)
kb['hs_code'] = kb['hs_code'].fillna('').astype(str)
kb['rule_type'] = kb['rule_type'].fillna('').astype(str)
kb['source_sheet'] = kb['source_sheet'].fillna('').astype(str)
kb['document_text_sw'] = kb['document_text'].map(lambda x:remove_stopword_morphs(x)[0])
data['ruling_sw'] = data[SUMMARY].map(lambda x:remove_stopword_morphs(x)[0])
data['ruling_removed_morphs'] = data[SUMMARY].map(lambda x:remove_stopword_morphs(x)[1])
kb['removed_morphs'] = kb['document_text'].map(lambda x:remove_stopword_morphs(x)[1])

print('판정요지 불용어 제거 형태소 수:',int(data['ruling_removed_morphs'].sum()))
print('판정요지 중 실제 변경된 사례 수:',int((data[SUMMARY]!=data['ruling_sw']).sum()),'/',len(data))
print('KB 불용어 제거 형태소 수:',int(kb['removed_morphs'].sum()))
print('KB 중 실제 변경된 문서 수:',int((kb['document_text']!=kb['document_text_sw']).sum()),'/',len(kb))
display(data.loc[data[SUMMARY]!=data['ruling_sw'],[TARGET,SUMMARY,'ruling_sw']].head(5))
display(kb.loc[kb['document_text']!=kb['document_text_sw'],['knowledge_id','document_text','document_text_sw']].head(5))

## 2. 공통 평가 함수 및 모델

모델은 08·09번과 동일한 `intfloat/multilingual-e5-base`를 사용합니다. 검색 질의에는 `query:`, 검색 문서에는 `passage:` 접두어를 붙입니다.

In [ ]:
MODEL_NAME = 'intfloat/multilingual-e5-base'
model = SentenceTransformer(MODEL_NAME)
def encode_docs(texts):
    return model.encode(['passage: '+str(x) for x in texts],batch_size=32,show_progress_bar=True,normalize_embeddings=True,convert_to_numpy=True)
def encode_query(text):
    return model.encode(['query: '+str(text)],normalize_embeddings=True,convert_to_numpy=True)[0]
def code_digits(value): return re.sub(r'\\D','',str(value))
def code_compatible(doc_code,true_code):
    a,b = code_digits(doc_code),code_digits(true_code)
    return bool(a and b and (a.startswith(b) or b.startswith(a)))
def retrieval_metrics(scores,relevant):
    order = np.argsort(scores)[::-1]
    ranked = np.asarray(relevant,dtype=bool)[order]
    hits = np.where(ranked)[0]
    out = {'mrr':1/(hits[0]+1) if len(hits) else 0.0}
    for k in [5,10]: out[f'recall@{k}'] = float(ranked[:k].sum()/max(int(np.sum(relevant)),1))
    gains = ranked[:10].astype(float)
    dcg = float((gains/np.log2(np.arange(2,len(gains)+2))).sum())
    ideal_n = min(int(np.sum(relevant)),10)
    idcg = float((np.ones(ideal_n)/np.log2(np.arange(2,ideal_n+2))).sum()) if ideal_n else 0.0
    out['ndcg@10'] = dcg/idcg if idcg else 0.0
    return out,np.argsort(scores)[::-1]

## 3. 실험 A — HS 지식베이스 문서 검색

지식베이스 문서의 본문에만 불용어를 적용합니다. 규칙 유형과 출처 시트 등 메타데이터는 유지합니다. 평가 제품의 다섯 속성으로 검색하고, HS 코드가 명시된 문서 중 상·하위 코드 접두어가 호환되는 문서를 관련 문서로 평가합니다.

In [ ]:
kb['search_text_base'] = (kb['document_text']+' '+kb['rule_type']+' '+kb['source_sheet']).str.strip()
kb['search_text_sw'] = (kb['document_text_sw']+' '+kb['rule_type']+' '+kb['source_sheet']).str.strip()
kb['is_code_specific'] = kb['hs_code'].map(code_digits).ne('')
doc_base = encode_docs(kb['search_text_base'].tolist())
doc_sw = encode_docs(kb['search_text_sw'].tolist())
query_embs = model.encode(['query: '+x for x in data['query_text']],batch_size=16,show_progress_bar=True,normalize_embeddings=True,convert_to_numpy=True)
kb_rows = []
for i,row in data.iterrows():
    relevant = np.array([code_compatible(x,row[TARGET]) for x in kb['hs_code']]) & kb['is_code_specific'].to_numpy()
    for variant,doc_emb in [('KB_original',doc_base),('KB_stopword',doc_sw)]:
        scores = doc_emb @ query_embs[i]
        met,order = retrieval_metrics(scores,relevant)
        kb_rows.append({'row_index':int(i),'true_hs':row[TARGET],'method':variant,**met,'relevant_doc_count':int(relevant.sum())})
kb_results = pd.DataFrame(kb_rows)
kb_summary = kb_results.groupby('method')[['mrr','ndcg@10','recall@5','recall@10']].agg(['mean','std']).round(4)
display(kb_summary)
display(kb_results.groupby('method')['relevant_doc_count'].agg(['mean','median','min','max']))

## 4. 실험 B — 과거 판정 사례 검색

검색 질의는 제품 속성 5개만 사용합니다. 검색 대상 과거 사례는 다음 세 조건으로 비교합니다.

- `product_only`: 과거 사례의 제품 속성만 사용
- `product_plus_ruling`: 제품 속성 + 원본 판정요지
- `product_plus_ruling_stopword`: 제품 속성 + 불용어를 제거한 판정요지

동일 제품 속성 중복이 학습·평가 Fold에 동시에 들어가지 않도록 4-fold StratifiedGroupKFold를 사용합니다.

In [ ]:
cv = StratifiedGroupKFold(n_splits=4,shuffle=True,random_state=42)
case_rows,fold_rows = [],[]
for fold,(tr,te) in enumerate(cv.split(data['query_text'],data[TARGET],groups=data['duplicate_group']),1):
    train,test = data.iloc[tr].copy(),data.iloc[te].copy()
    overlap = len(set(train['duplicate_group'])&set(test['duplicate_group']))
    fold_rows.append({'fold':fold,'train_n':len(train),'test_n':len(test),'group_overlap':overlap})
    docs = {
        'product_only':train['query_text'].tolist(),
        'product_plus_ruling':(train['query_text']+' '+train[SUMMARY]).str.strip().tolist(),
        'product_plus_ruling_stopword':(train['query_text']+' '+train['ruling_sw']).str.strip().tolist()
    }
    embeddings = {name:encode_docs(texts) for name,texts in docs.items()}
    for idx,row in test.iterrows():
        q = encode_query(row['query_text'])
        for method,doc_emb in embeddings.items():
            scores = doc_emb @ q
            relevant = train[TARGET].eq(row[TARGET]).to_numpy()
            met,order = retrieval_metrics(scores,relevant)
            case_rows.append({'fold':fold,'row_index':int(idx),'true_hs':row[TARGET],'method':method,**met,'top1_hs':train.iloc[order[0]][TARGET]})
case_results = pd.DataFrame(case_rows)
fold_audit = pd.DataFrame(fold_rows)
case_summary = case_results.groupby('method')[['mrr','ndcg@10','recall@5','recall@10']].agg(['mean','std']).round(4)
display(case_summary)
display(fold_audit)
print('Fold 그룹 중복 합계:',int(fold_audit['group_overlap'].sum()))

## 5. 불용어 적용 효과 및 결과 저장

검색 지표의 차이는 동일 제품별 결과를 기준으로 계산합니다. 평균 차이뿐 아니라 제품별로 검색 순위가 개선·악화된 사례도 확인합니다. 지식베이스 검색과 과거 사례 검색은 검색 대상과 관련 문서 정의가 다르므로 두 실험의 지표를 서로 직접 비교하지 않습니다.

In [ ]:
def paired_comparison(frame,base,variant,metrics):
    a = frame[frame['method'].eq(base)].sort_values('row_index').reset_index(drop=True)
    b = frame[frame['method'].eq(variant)].sort_values('row_index').reset_index(drop=True)
    rows = []
    for metric in metrics:
        diff = b[metric]-a[metric]
        rows.append({'comparison':f'{base} -> {variant}','metric':metric,'base_mean':a[metric].mean(),'variant_mean':b[metric].mean(),'mean_difference':diff.mean(),'improved_n':int((diff>0).sum()),'worsened_n':int((diff<0).sum()),'unchanged_n':int((diff==0).sum())})
    return rows
metrics = ['mrr','ndcg@10','recall@5','recall@10']
comparisons = paired_comparison(kb_results,'KB_original','KB_stopword',metrics)
comparisons += paired_comparison(case_results,'product_plus_ruling','product_plus_ruling_stopword',metrics)
comparison_table = pd.DataFrame(comparisons)
display(comparison_table.round(4))
kb_results.to_csv('16_kb_stopword_retrieval.csv',index=False,encoding='utf-8-sig')
case_results.to_csv('16_case_stopword_retrieval.csv',index=False,encoding='utf-8-sig')
comparison_table.to_csv('16_stopword_comparison.csv',index=False,encoding='utf-8-sig')
fold_audit.to_csv('16_fold_audit.csv',index=False,encoding='utf-8-sig')
print('16번 결과 저장 완료')